In [ ]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

In [ ]:
if IN_COLAB:
  # Install dependencies
  ! pip install --upgrade pip
  ! pip install czitools[ndv]
  ! pip install ipyfilechooser

In [ ]:
# import the required libraries
from czitools.metadata_tools import czi_metadata
from czitools.utils import misc
from czitools.read_tools import read_tools
from ipyfilechooser import FileChooser
from IPython.display import display, HTML
import os
import dask.array as da
import requests
import glob
import ipywidgets as widgets
import ndv
import xarray as xr
#from typing import Any, cast

In [ ]:
# try to find the folder with data and download otherwise from GitHub.

# Folder containing the input data
if IN_COLAB:
    INPUT_FOLDER = 'data/'
if not IN_COLAB:
    INPUT_FOLDER = '../../data/'

# Path to the data on GitHub
GITHUB_IMAGES_PATH = "https://raw.githubusercontent.com/sebi06/czitools/main/data.zip"

# Download data
if not (os.path.isdir(INPUT_FOLDER)):
    compressed_data = './data.zip'
    if not os.path.isfile(compressed_data):
        import io
        response = requests.get(GITHUB_IMAGES_PATH, stream=True)
        compressed_data = io.BytesIO(response.content)

    import zipfile
    with zipfile.ZipFile(compressed_data, 'r') as zip_accessor:
        zip_accessor.extractall('./')

In [ ]:
if not IN_COLAB:
    # choose local file
    fc = FileChooser()
    fc.default_path = INPUT_FOLDER
    fc.filter_pattern = '*.czi'
    display(fc)

elif IN_COLAB:
    # list files inside the folder on gdrive
    czifiles = glob.glob(os.path.join(INPUT_FOLDER, "*.czi"))
    wd = widgets.Select(
        options=czifiles,
        description='CZI Files:',
        layout={'width': 'max-content'}
    )
    display(wd)

In [ ]:
if not IN_COLAB:
    filepath = fc.selected
elif IN_COLAB:
    filepath = wd.value

print(f"Selected File: {filepath}")

In [ ]:
# get the complete metadata at once as one big class
mdata = czi_metadata.CziMetadata(filepath)

# convert metadata dictionary to a pandas dataframe
mdframe = misc.md2dataframe(mdata, reduced_params=True)

# create a ipywdiget to show the dataframe with the metadata
wd = widgets.Output(layout={"scrollY": "auto", "height": "300px"})

with wd:
    display(HTML(mdframe.to_html()))
display(widgets.VBox(children=[wd]))

In [ ]:
result, dims, num_stacks, mdata = read_tools.read_stacks(
        filepath,
        zoom=1.0,
        use_dask=True,  # use lazy dask arrays (data read only when accessed)
        use_xarray=True,  # return xarray DataArray with labeled dimensions (STCZYX(A))
        stack_scenes=True,  # stack scenes with the same shape into one array (if False, return a list of arrays)
        adapt_metadata=True,  # adapt metadata according to the planes specified (SizeS, SizeT, SizeC, SizeZ)
    )

print("\n=== Results ===")
print(f"Array Shape: {result.shape}")
print(f"Number of stacks: {num_stacks}")
print(f"Dimension order: {dims}")

if isinstance(result, list):
    # List of per-scene arrays
    for idx, arr in enumerate(result):
        if isinstance(arr, xr.DataArray):
            print(f"Stack {idx}: dims={arr.dims}, shape={arr.shape}, dtype={arr.dtype}")
        else:
            print(f"Stack {idx}: shape={arr.shape}, dims={dims}, dtype={arr.dtype}")
else:
    # Single stacked array
    if isinstance(result, xr.DataArray):
        print(f"Stacked: dims={result.dims}, shape={result.shape}, dtype={result.dtype}")
    else:
        print(f"Stacked: shape={result.shape}, dims={dims}, dtype={result.dtype}")

    # With use_dask=True, result is backed by dask - no data loaded yet
    print(f"\nArray shape (no data loaded): {result.shape}")

In [ ]:
# verify whether result/results is a Dask array and display its structure
candidate = globals().get("results", globals().get("result"))

if candidate is None:
    print("No variable named 'result' or 'results' found yet.")
    print("Run the read_stacks cell first, then run this cell again.")
elif isinstance(candidate, xr.DataArray):
    is_dask = isinstance(candidate.data, da.Array)
    print(f"xarray.DataArray detected: {type(candidate)}")
    print(f"Backed by Dask: {is_dask}")
    if is_dask:
        darr = candidate.data
        print("\nDask-backed structure:")
        print(darr)
        print(f"shape   : {darr.shape}")
        print(f"dtype   : {darr.dtype}")
        print(f"chunks  : {darr.chunks}")
        print(f"numblocks: {darr.numblocks}")
        print(f"npartitions: {darr.npartitions}")
    else:
        print(f"shape: {candidate.shape}, dtype: {candidate.dtype}")
elif isinstance(candidate, da.Array):
    print("Dask array detected")
    print(candidate)
    print(f"shape   : {candidate.shape}")
    print(f"dtype   : {candidate.dtype}")
    print(f"chunks  : {candidate.chunks}")
    print(f"numblocks: {candidate.numblocks}")
    print(f"npartitions: {candidate.npartitions}")
else:
    print(f"Not a Dask array. Type: {type(candidate)}")
    if hasattr(candidate, "shape") and hasattr(candidate, "dtype"):
        print(f"shape: {candidate.shape}, dtype: {candidate.dtype}")

In [ ]:
# visualize array structure
result

In [ ]:
# NOTE: NDV Jupyter backend currently expects slider coordinates as `range` objects.
# Passing an xarray DataArray directly can fail when coordinates are not ranges.

# Use raw array backing data to avoid xarray coordinate handling in NDV Jupyter view.
viewer_data = result.data if hasattr(result, "data") else result

# NOTE: This does not work yet

# Display the data in NDV with scale metadata.
#viewer = ndv.imshow(viewer_data, channel_mode="composite", channel_axis=2)